In [ ]:
import os, re, zipfile, shutil, torch
from PIL import Image, ImageFilter
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import DonutProcessor, VisionEncoderDecoderModel
from peft import LoraConfig, get_peft_model, TaskType
#1000장, 5epoch
# [1. 설정]
BASE_PATH = "/content/drive/MyDrive/인지응 3팀 공유폴더/1000장"
ZIP_PATH = f"{BASE_PATH}/solo.zip"
LABEL_FILE = f"{BASE_PATH}/label_log.txt"
SAVE_PATH = f"{BASE_PATH}/donut_snack_final_5epoch"
IMAGE_FINAL_DIR = "/content/dataset/images"

# [2. 전처리 및 이미지 추출]
if os.path.exists(IMAGE_FINAL_DIR): shutil.rmtree(IMAGE_FINAL_DIR)
os.makedirs(IMAGE_FINAL_DIR, exist_ok=True)

print("1,000장 이미지 전처리 및 추출 시작...")
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    all_files = z.namelist()
    for i in range(1, 1001):
        target_folder = f"sequence.{i}/"
        match = [f for f in all_files if target_folder in f and f.endswith("step0.camera.png")]
        if match:
            with z.open(match[0]) as f:
                img = Image.open(f).convert("RGB")
                img = img.filter(ImageFilter.MedianFilter(size=3))
                img = img.resize((640, 640), Image.LANCZOS)
                img.save(os.path.join(IMAGE_FINAL_DIR, f"sequence.{i}.png"))
        if i % 200 == 0: print(f"진행율: {i}/1000")

# [3. 데이터셋 및 모델 설정]
class SnackDataset(Dataset):
    def __init__(self, img_dir, label_file, processor):
        self.img_dir, self.processor = img_dir, processor
        with open(label_file, "r", encoding="utf-8") as f:
            self.lines = [l.strip().replace("Ingredients:", "").strip() for l in f if l.strip()]
        existing_imgs = os.listdir(img_dir)
        self.img_files = [f"sequence.{i+1}.png" for i in range(len(self.lines)) if f"sequence.{i+1}.png" in existing_imgs]
    def __len__(self): return len(self.img_files)
    def __getitem__(self, idx):
        img_name = self.img_files[idx]
        img = Image.open(os.path.join(self.img_dir, img_name))
        line_idx = int(re.findall(r'\d+', img_name)[0]) - 1
        target = f"<s><s_food><s_원재료>{self.lines[line_idx]}</s_원재료></s_food></s>"
        pixel_values = self.processor(img, return_tensors="pt").pixel_values.squeeze()
        labels = self.processor.tokenizer(target, add_special_tokens=False, max_length=256, padding="max_length", truncation=True, return_tensors="pt").input_ids.squeeze()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base")
processor.tokenizer.add_tokens(["<s_food>", "</s_food>", "<s_원재료>", "</s_원재료>"])
model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base", device_map="auto", torch_dtype=torch.float16)
model.decoder.resize_token_embeddings(len(processor.tokenizer))
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids("<s_food>")

config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.1, task_type=TaskType.SEQ_2_SEQ_LM)
model = get_peft_model(model, config)

EPOCHS = 5
BATCH_SIZE = 2
accumulation_steps = 4

dataset = SnackDataset(IMAGE_FINAL_DIR, LABEL_FILE, processor)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

model.train()
print(f" {BATCH_SIZE * accumulation_steps})")

for epoch in range(EPOCHS):
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    opt.zero_grad()

    for i, b in enumerate(pbar):
        pixel_values = b["pixel_values"].to("cuda", dtype=torch.float16)
        labels = b["labels"].to("cuda")

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        loss = loss / accumulation_steps
        loss.backward()
        if (i + 1) % accumulation_steps == 0:
            opt.step()
            opt.zero_grad()

        pbar.set_postfix(loss=loss.item() * accumulation_steps)

    epoch_save_path = f"{SAVE_PATH}_epoch_{epoch+1}"
    if hasattr(model.config, "vocab_size"): del model.config.vocab_size
    model.save_pretrained(epoch_save_path)
    processor.save_pretrained(epoch_save_path)
    print(f"\n📍 Epoch {epoch+1} 완료 및 저장됨!")